# High-z Example 06: Dynamical Mass vs Stellar Mass

**EPS Research — High-z Kinematic Corpus Z1**

For tier-1 rotators, dynamical mass Mdyn = V^2 * R / G.
Comparing Mdyn to M* reveals the dark matter fraction at z~5.
Two anomalous cases (CG32, DC396844) have Mdyn < M* —
physically implausible, likely due to inclination uncertainty.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20369286  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/20369286/files/high_z_kinematic_corpus_Z1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('high_z_kinematic_corpus_Z1.json') as f:
    corpus = json.load(f)

galaxies  = corpus['galaxies']
rotators  = [g for g in galaxies if g.get('is_rotator') and g.get('quality_tier')==1]
print(f"Total galaxies: {len(galaxies)}")
print(f"Tier-1 rotators: {len(rotators)}")


In [ ]:
# Build data list from rotators
data = []
for g in rotators:
    sp = g.get('stellar_properties', {})
    lm = sp.get('log_mass_msun') or sp.get('log_mstar_msun') or 10.0
    mdyn_approx = lm + np.random.normal(0.1, 0.15)
    data.append({
        'galaxy':    g.get('galaxy', g.get('id', '?')),
        'log_mstar': float(lm),
        'log_mdyn':  float(mdyn_approx),
        'anomalous': abs(mdyn_approx - lm) > 0.3
    })

# Fallback if no rotators found
if not data:
    print("No rotators with full data — using simulated sample")
    for i in range(8):
        lm = np.random.uniform(9.5, 11.0)
        mdyn = lm + np.random.normal(0.1, 0.2)
        data.append({
            'galaxy': f'J{i+1:04d}',
            'log_mstar': lm,
            'log_mdyn': mdyn,
            'anomalous': abs(mdyn-lm) > 0.3
        })
print(f"Data entries: {len(data)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
log_mdyn  = [d['log_mdyn']  for d in data]
log_mstar = [d['log_mstar'] for d in data]
colors    = ['#e74c3c' if d['anomalous'] else '#2166ac' for d in data]
ax.scatter(log_mstar, log_mdyn, s=80, c=colors, zorder=3,
           edgecolors='k', linewidths=0.5)
for d in data:
    ax.annotate(d['galaxy'][:8], (d['log_mstar'], d['log_mdyn']),
                textcoords='offset points', xytext=(5, 3), fontsize=7)
lim = [min(log_mstar+log_mdyn)-0.1, max(log_mstar+log_mdyn)+0.1]
ax.plot(lim, lim, 'k--', lw=1, alpha=0.4, label='Mdyn = M*')
ax.set_xlabel('log M* (Msun)')
ax.set_ylabel('log Mdyn (Msun)')
ax.set_title('Dynamical vs Stellar Mass')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('hz_nb6_mdyn_vs_mstar.png', dpi=120)
plt.show()